# 08_3 DQA Scene Phase2 Update-Gating Sweep 7h

08/08_2 の結果から、phase2 は pseudoGT の選別だけではなく **target client update の許容量** を制御する必要がありそうです。この notebook は 08 phase1 checkpoint から phase2 だけを短く回し、DQA らしい update gating を 5 パターン比較します。

各 variant は 10 round。08_2 の実測が約 7.9 分/round なので、5 本合計で約 6.5 時間、setup/eval を含めて 7 時間程度を狙います。


## 1. Paths

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import socket
import subprocess
import sys
import time
from collections import deque
from pathlib import Path
from typing import Optional

import pandas as pd


def find_repo_root(start: Optional[Path] = None) -> Path:
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    required = (
        "dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2_phase2_update_gating_sweep.py",
        "dynamic_quality_aware_classwise_aggregation/evaluate_scene_protocol.py",
        "navigating_data_heterogeneity/setup_fedsto_scene_reproduction.py",
    )
    for base in (start, *start.parents):
        for candidate in (base, base / "Object_Detection"):
            if all((candidate / marker).exists() for marker in required):
                return candidate.resolve()
    raise FileNotFoundError("Could not locate /app/Object_Detection")


REPO_ROOT = find_repo_root()
DQA_ROOT = REPO_ROOT / "dynamic_quality_aware_classwise_aggregation"
RUN_SCRIPT = DQA_ROOT / "run_dqa_cwa_fedsto_scene_v2_phase2_update_gating_sweep.py"
EVAL_SCRIPT = DQA_ROOT / "evaluate_scene_protocol.py"
POLICY_MODEL = DQA_ROOT / "threshold_policy_model" / "artifacts" / "dqa05_threshold_policy.joblib"
SOURCE_WORK_ROOT = DQA_ROOT / "efficientteacher_dqa08_scene_tri_stage_policy_8h"
BASE_WORK_ROOT = DQA_ROOT / "efficientteacher_dqa08_3_phase2_update_gating_sweep"
BASE_STATS_ROOT = DQA_ROOT / "stats_dqa08_3_phase2_update_gating_sweep"
BASE_LOG_ROOT = DQA_ROOT / "logs_dqa08_3_phase2_update_gating_sweep"

preferred_python = Path("/root/micromamba/envs/al_yolov8/bin/python")
PYTHON_BIN = preferred_python if preferred_python.exists() else Path(sys.executable)

print("repo_root:", REPO_ROOT)
print("source_08_workspace:", SOURCE_WORK_ROOT)
print("base_workspace:", BASE_WORK_ROOT)
print("base_stats:", BASE_STATS_ROOT)
print("python:", PYTHON_BIN)
print("policy_model:", POLICY_MODEL)


## 2. Five DQA Phase2 Patterns

In [ ]:
PHASE2_ROUNDS_PER_VARIANT = 10
WARMUP_EPOCHS = 0
PHASE1_ROUNDS = 0
DQA_START_PHASE = 2

BATCH_SIZE = 160
WORKERS = 8
REQUESTED_GPUS = 2
MIN_FREE_GIB = 8

# 08_2 was about 7.9 min/round. 10 rounds x 5 variants is about 395 min.
EST_MIN_PER_ROUND = 7.9

DEFAULT_POLICY = {
    "adapt_start_round": 2,
    "min_low": 0.38,
    "max_low": 0.48,
    "min_mid": 0.58,
    "max_mid": 0.72,
    "min_high": 0.84,
    "max_high": 0.92,
    "low_shift": 0.02,
    "high_shift": 0.05,
    "mid_gap": 0.20,
    "high_mid_gap": 0.18,
    "min_high_gap": 0.16,
    "max_nms": 0.45,
    "teacher_min": 0.28,
    "rare_count": 250,
    "rare_max_low": 0.43,
    "rare_max_mid": 0.66,
    "rare_max_high": 0.89,
    "low_step_limit": 0.02,
    "mid_step_limit": 0.025,
    "high_step_limit": 0.035,
    "nms_step_limit": 0.02,
    "low_mid_obj_weight": 0.45,
    "mid_high_obj_weight": 1.0,
}

VARIANTS = [
    {
        "name": "a_ug_decay_backbone_r012",
        "description": "本命。backbone-only + uncertain ignore + DQA residual 0.12->0.04 + strong source floor。",
        "source_phase1_round": 12,
        "client_train_scope": "backbone",
        "uncertain_ignore": True,
        "residual_start": 0.12,
        "residual_end": 0.04,
        "min_server_alpha": 0.75,
        "classwise_blend": 0.10,
        "server_anchor": 8.0,
        "temperature": 2.0,
        "stability_lambda": 0.40,
    },
    {
        "name": "b_ug_tiny_anchor_r012",
        "description": "precision 保護寄り。client residual をかなり小さくし、source/server をさらに強くする。",
        "source_phase1_round": 12,
        "client_train_scope": "backbone",
        "uncertain_ignore": True,
        "residual_start": 0.06,
        "residual_end": 0.02,
        "min_server_alpha": 0.85,
        "classwise_blend": 0.06,
        "server_anchor": 12.0,
        "temperature": 2.5,
        "stability_lambda": 0.55,
    },
    {
        "name": "c_ug_all_tiny_high_r012",
        "description": "high pseudoGT の bbox/head 学習を許すが、aggregation residual は極小にする。",
        "source_phase1_round": 12,
        "client_train_scope": "all",
        "uncertain_ignore": True,
        "residual_start": 0.05,
        "residual_end": 0.02,
        "min_server_alpha": 0.85,
        "classwise_blend": 0.05,
        "server_anchor": 12.0,
        "temperature": 2.5,
        "stability_lambda": 0.55,
    },
    {
        "name": "d_ug_strict_high_r012",
        "description": "pseudoGT precision 優先。high gate をさらに厳しくして、少ないが安定した target update だけ許す。",
        "source_phase1_round": 12,
        "client_train_scope": "backbone",
        "uncertain_ignore": True,
        "residual_start": 0.10,
        "residual_end": 0.03,
        "min_server_alpha": 0.80,
        "classwise_blend": 0.08,
        "server_anchor": 10.0,
        "temperature": 2.2,
        "stability_lambda": 0.50,
        "policy": {
            "min_low": 0.42,
            "max_low": 0.52,
            "min_mid": 0.68,
            "max_mid": 0.80,
            "min_high": 0.90,
            "max_high": 0.96,
            "rare_max_low": 0.47,
            "rare_max_mid": 0.74,
            "rare_max_high": 0.93,
            "max_nms": 0.40,
        },
    },
    {
        "name": "e_ug_best_phase1_r003",
        "description": "phase1 の best checkpoint から同じ update gating を試す。seed が悪化要因かを見る。",
        "source_phase1_round": 3,
        "client_train_scope": "backbone",
        "uncertain_ignore": True,
        "residual_start": 0.10,
        "residual_end": 0.03,
        "min_server_alpha": 0.75,
        "classwise_blend": 0.10,
        "server_anchor": 8.0,
        "temperature": 2.0,
        "stability_lambda": 0.40,
    },
]

# 空なら全 variant 実行。必要なら ["a_ug_decay_backbone_r012"] のように絞る。
SELECTED_VARIANTS: list[str] = []
RUN_TRAINING = True
RUN_IN_BACKGROUND = False
STREAM_TRAIN_OUTPUT = True
APPEND_TRAIN_LOG = False

try:
    import torch

    AVAILABLE_CUDA_GPUS = torch.cuda.device_count()
except Exception as exc:
    AVAILABLE_CUDA_GPUS = 0
    print("Could not inspect CUDA devices:", exc)

GPUS = min(REQUESTED_GPUS, AVAILABLE_CUDA_GPUS) if AVAILABLE_CUDA_GPUS else 1
if GPUS != REQUESTED_GPUS:
    print(f"Requested {REQUESTED_GPUS} GPU(s), visible={AVAILABLE_CUDA_GPUS}; using GPUS={GPUS}")


def find_free_port(preferred: int) -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        try:
            sock.bind(("127.0.0.1", preferred))
            return preferred
        except OSError:
            pass
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return int(sock.getsockname()[1])


MASTER_PORT = find_free_port(29610)
selected = [v for v in VARIANTS if not SELECTED_VARIANTS or v["name"] in SELECTED_VARIANTS]

summary = pd.DataFrame(
    [
        {
            "name": v["name"],
            "seed_r": v["source_phase1_round"],
            "scope": v["client_train_scope"],
            "residual": f"{v['residual_start']}->{v['residual_end']}",
            "server_floor": v["min_server_alpha"],
            "classwise_blend": v["classwise_blend"],
            "server_anchor": v["server_anchor"],
            "description": v["description"],
        }
        for v in selected
    ]
)
display(summary)
print("estimated_training_minutes:", round(len(selected) * PHASE2_ROUNDS_PER_VARIANT * EST_MIN_PER_ROUND, 1))
print("master_port:", MASTER_PORT)
print("gpus:", GPUS)


## 3. Helpers

In [ ]:
def variant_work_root(variant: dict) -> Path:
    return BASE_WORK_ROOT / variant["name"]


def variant_stats_root(variant: dict) -> Path:
    return BASE_STATS_ROOT / variant["name"]


def variant_runner_log(variant: dict) -> Path:
    return BASE_LOG_ROOT / f"{variant['name']}_runner.out"


def variant_train_log(variant: dict) -> Path:
    return BASE_LOG_ROOT / f"{variant['name']}_train.log"


def variant_pid_path(variant: dict) -> Path:
    return BASE_LOG_ROOT / f"{variant['name']}.pid"


def policy_value(variant: dict, key: str):
    return dict(DEFAULT_POLICY, **variant.get("policy", {}))[key]


def variant_env(variant: dict) -> dict[str, str]:
    env = os.environ.copy()
    stats_root = variant_stats_root(variant)
    threshold_log = stats_root / "update_gating_policy_schedule.jsonl"

    env["DQA08_2_SOURCE_WORK_ROOT"] = str(SOURCE_WORK_ROOT)
    env["DQA08_2_SOURCE_PHASE1_ROUND"] = str(variant["source_phase1_round"])
    env["DQA08_2_FORCE_SEED"] = "0"
    env["DQA08_2_CLIENT_TRAIN_SCOPE"] = variant["client_train_scope"]
    env["DQA08_2_SERVER_TRAIN_SCOPE"] = "all"
    env["DQA08_2_UNCERTAIN_IGNORE"] = "1" if variant["uncertain_ignore"] else "0"
    env["DQA08_2_CLIENT_LR0"] = "0.0003"
    env["DQA08_2_SERVER_ORTHOGONAL_WEIGHT"] = "0.0001"

    env["DQA08_3_VARIANT"] = variant["name"]
    env["DQA08_3_RESIDUAL_START"] = str(variant["residual_start"])
    env["DQA08_3_RESIDUAL_END"] = str(variant["residual_end"])
    env["DQA08_3_MIN_SERVER_ALPHA"] = str(variant["min_server_alpha"])
    env["DQA08_3_PHASE2_ROUNDS"] = str(PHASE2_ROUNDS_PER_VARIANT)

    env["DQA08_SSOD_PROFILE"] = "dqa08_3_update_gating_sweep"
    env["DQA08_POLICY_MODEL"] = str(POLICY_MODEL)
    env["DQA08_POLICY_HORIZON_ROUNDS"] = str(PHASE2_ROUNDS_PER_VARIANT)
    env["DQA08_CLIENT_LR0"] = "0.0003"
    env["DQA08_SERVER_LR0"] = "0.001"
    env["DQA08_ADAPT_START_ROUND"] = str(policy_value(variant, "adapt_start_round"))
    env["DQA08_MIN_LOW"] = str(policy_value(variant, "min_low"))
    env["DQA08_MAX_LOW"] = str(policy_value(variant, "max_low"))
    env["DQA08_MIN_MID"] = str(policy_value(variant, "min_mid"))
    env["DQA08_MAX_MID"] = str(policy_value(variant, "max_mid"))
    env["DQA08_MIN_HIGH"] = str(policy_value(variant, "min_high"))
    env["DQA08_MAX_HIGH"] = str(policy_value(variant, "max_high"))
    env["DQA08_LOW_SHIFT"] = str(policy_value(variant, "low_shift"))
    env["DQA08_HIGH_SHIFT"] = str(policy_value(variant, "high_shift"))
    env["DQA08_MID_GAP"] = str(policy_value(variant, "mid_gap"))
    env["DQA08_HIGH_MID_GAP"] = str(policy_value(variant, "high_mid_gap"))
    env["DQA08_MIN_HIGH_GAP"] = str(policy_value(variant, "min_high_gap"))
    env["DQA08_MAX_NMS"] = str(policy_value(variant, "max_nms"))
    env["DQA08_TEACHER_MIN"] = str(policy_value(variant, "teacher_min"))
    env["DQA08_RARE_COUNT"] = str(policy_value(variant, "rare_count"))
    env["DQA08_RARE_MAX_LOW"] = str(policy_value(variant, "rare_max_low"))
    env["DQA08_RARE_MAX_MID"] = str(policy_value(variant, "rare_max_mid"))
    env["DQA08_RARE_MAX_HIGH"] = str(policy_value(variant, "rare_max_high"))
    env["DQA08_LOW_STEP_LIMIT"] = str(policy_value(variant, "low_step_limit"))
    env["DQA08_MID_STEP_LIMIT"] = str(policy_value(variant, "mid_step_limit"))
    env["DQA08_HIGH_STEP_LIMIT"] = str(policy_value(variant, "high_step_limit"))
    env["DQA08_NMS_STEP_LIMIT"] = str(policy_value(variant, "nms_step_limit"))
    env["DQA08_LOW_MID_OBJ_WEIGHT"] = str(policy_value(variant, "low_mid_obj_weight"))
    env["DQA08_MID_HIGH_OBJ_WEIGHT"] = str(policy_value(variant, "mid_high_obj_weight"))
    env["DQA08_THRESHOLD_LOG"] = str(threshold_log)
    return env


def train_cmd(variant: dict, *, stream: bool = STREAM_TRAIN_OUTPUT) -> list[str]:
    cmd = [
        str(PYTHON_BIN),
        "-u",
        str(RUN_SCRIPT),
        "--workspace-root",
        str(variant_work_root(variant)),
        "--stats-root",
        str(variant_stats_root(variant)),
        "--warmup-epochs",
        str(WARMUP_EPOCHS),
        "--phase1-rounds",
        str(PHASE1_ROUNDS),
        "--phase2-rounds",
        str(PHASE2_ROUNDS_PER_VARIANT),
        "--dqa-start-phase",
        str(DQA_START_PHASE),
        "--batch-size",
        str(BATCH_SIZE),
        "--workers",
        str(WORKERS),
        "--gpus",
        str(GPUS),
        "--master-port",
        str(MASTER_PORT),
        "--min-free-gib",
        str(MIN_FREE_GIB),
        "--log-file",
        str(variant_train_log(variant)),
        "--classwise-blend",
        str(variant["classwise_blend"]),
        "--server-anchor",
        str(variant["server_anchor"]),
        "--temperature",
        str(variant["temperature"]),
        "--stability-lambda",
        str(variant["stability_lambda"]),
        "--localize-bn",
        "--enable-dqa-guard",
        "--dqa-drop-ratio-threshold",
        "0.15",
        "--dqa-spike-ratio-threshold",
        "3.0",
    ]
    if APPEND_TRAIN_LOG:
        cmd.append("--append-train-log")
    if stream:
        cmd.append("--stream-train-output")
    return cmd


def read_pid(path: Path) -> int | None:
    if not path.exists():
        return None
    try:
        return int(path.read_text(encoding="utf-8").strip())
    except ValueError:
        return None


def pid_state(pid: int | None) -> str:
    if pid is None:
        return "missing"
    result = subprocess.run(["ps", "-o", "stat=", "-p", str(pid)], capture_output=True, text=True)
    state = result.stdout.strip()
    if result.returncode != 0 or not state:
        return "missing"
    if "Z" in state:
        return "zombie"
    return state


def history_rows(variant: dict) -> list[dict]:
    path = variant_work_root(variant) / "history.json"
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else []


## 4. Build Lists

In [ ]:
if not POLICY_MODEL.exists():
    raise FileNotFoundError(f"Policy model is missing: {POLICY_MODEL}")

BASE_LOG_ROOT.mkdir(parents=True, exist_ok=True)
for variant in selected:
    subprocess.run(
        [
            str(PYTHON_BIN),
            str(RUN_SCRIPT),
            "--setup-only",
            "--workspace-root",
            str(variant_work_root(variant)),
            "--stats-root",
            str(variant_stats_root(variant)),
        ],
        cwd=REPO_ROOT,
        check=True,
        env=variant_env(variant),
    )

manifest = json.loads((variant_work_root(selected[0]) / "manifest.json").read_text(encoding="utf-8")) if selected else {}
if manifest:
    display(pd.DataFrame([manifest["server"]]))
    display(pd.DataFrame(manifest["clients"]))
    display(pd.DataFrame(manifest["paper_evaluation"]["splits"])[["name", "raw_scene", "images", "boxes"]])

print("prepared variants:", [v["name"] for v in selected])


## 5. Command Preview

In [ ]:
for variant in selected:
    print("\n===", variant["name"], "===")
    print(variant["description"])
    print("workspace:", variant_work_root(variant))
    print("stats:", variant_stats_root(variant))
    print(" ".join(train_cmd(variant, stream=False)))


## 6. Run Five Variants

In [ ]:
start_time = time.time()

for variant in selected:
    history = history_rows(variant)
    completed_phase2 = sum(1 for row in history if int(row.get("phase", 0)) == 2)
    print("\n" + "=" * 100)
    print("variant:", variant["name"])
    print(variant["description"])
    print(f"completed_phase2: {completed_phase2}/{PHASE2_ROUNDS_PER_VARIANT}")
    if completed_phase2 >= PHASE2_ROUNDS_PER_VARIANT:
        print("Already complete, skipping.")
        continue

    pid_path = variant_pid_path(variant)
    current_pid = read_pid(pid_path)
    state = pid_state(current_pid)
    if RUN_TRAINING and state not in {"missing", "zombie"}:
        print("Training already appears to be running:", current_pid, state)
        continue
    if not RUN_TRAINING:
        print("RUN_TRAINING=False, command was not launched.")
        continue

    env = variant_env(variant)
    cmd = train_cmd(variant)
    runner_log = variant_runner_log(variant)
    runner_log.parent.mkdir(parents=True, exist_ok=True)
    log_mode = "a" if APPEND_TRAIN_LOG else "w"
    print("command:", " ".join(cmd))
    print("runner_log:", runner_log)
    print("train_log:", variant_train_log(variant))

    if RUN_IN_BACKGROUND:
        with runner_log.open("ab" if APPEND_TRAIN_LOG else "wb") as out:
            process = subprocess.Popen(
                cmd,
                cwd=REPO_ROOT,
                stdout=out,
                stderr=subprocess.STDOUT,
                env=env,
                start_new_session=True,
            )
        pid_path.write_text(str(process.pid), encoding="utf-8")
        print("Started PID:", process.pid)
    else:
        with runner_log.open(log_mode, encoding="utf-8", buffering=1) as out:
            process = subprocess.Popen(
                cmd,
                cwd=REPO_ROOT,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
                env=env,
            )
            pid_path.write_text(str(process.pid), encoding="utf-8")
            print("Started PID:", process.pid)
            out.write(f"Started PID: {process.pid}\n")
            assert process.stdout is not None
            for line in process.stdout:
                print(line, end="")
                out.write(line)
            return_code = process.wait()
        if pid_path.exists() and pid_path.read_text(encoding="utf-8").strip() == str(process.pid):
            pid_path.unlink()
        if return_code != 0:
            raise RuntimeError(f"{variant['name']} failed with exit code {return_code}. See {runner_log}")
        print("variant completed:", variant["name"])

elapsed_min = (time.time() - start_time) / 60
print("elapsed_minutes:", round(elapsed_min, 1))


## 7. Status

In [ ]:
def tail_lines(path: Path, lines: int = 25) -> list[str]:
    if not path.exists():
        return []
    try:
        result = subprocess.run(["tail", "-n", str(lines), str(path)], capture_output=True, text=True, check=True)
        return result.stdout.splitlines()
    except Exception:
        with path.open(encoding="utf-8", errors="replace") as f:
            return [line.rstrip("\n") for line in deque(f, maxlen=lines)]

rows = []
for variant in selected:
    history = history_rows(variant)
    latest = Path(history[-1]["global"]) if history else variant_work_root(variant) / "global_checkpoints" / "round000_warmup.pt"
    rows.append(
        {
            "variant": variant["name"],
            "pid": read_pid(variant_pid_path(variant)),
            "pid_state": pid_state(read_pid(variant_pid_path(variant))),
            "phase2": f"{sum(1 for row in history if int(row.get('phase', 0)) == 2)}/{PHASE2_ROUNDS_PER_VARIANT}",
            "latest_global": str(latest),
            "exists": latest.exists(),
            "free_gib": round(shutil.disk_usage(variant_work_root(variant)).free / 1024**3, 2) if variant_work_root(variant).exists() else None,
        }
    )
display(pd.DataFrame(rows))

for variant in selected:
    print("\n===", variant["name"], "runner tail ===")
    for line in tail_lines(variant_runner_log(variant), 20):
        print(line)


## 8. Evaluate Final Checkpoints

In [ ]:
RUN_EVAL = False
EVAL_SPLITS = "highway,citystreet,residential,total"
EVAL_BATCH_SIZE = 16
EVAL_DEVICE = ""

if not selected:
    raise RuntimeError("No selected variants.")

EVAL_WORKSPACE = variant_work_root(selected[0])
checkpoints: list[tuple[str, Path]] = []
seed12 = SOURCE_WORK_ROOT / "global_checkpoints" / "phase1_round012_global.pt"
seed03 = SOURCE_WORK_ROOT / "global_checkpoints" / "phase1_round003_global.pt"
old08 = SOURCE_WORK_ROOT / "global_checkpoints" / "phase2_round024_global.pt"
old082 = DQA_ROOT / "efficientteacher_dqa08_2_scene_phase2_head_protected" / "global_checkpoints" / "phase2_round024_global.pt"
for label, path in [
    ("dqa08_phase1_r012", seed12),
    ("dqa08_phase1_r003", seed03),
    ("dqa08_phase2_r024", old08),
    ("dqa08_2_phase2_r024", old082),
]:
    if path.exists():
        checkpoints.append((label, path))

for variant in selected:
    history = history_rows(variant)
    phase2 = [row for row in history if int(row.get("phase", 0)) == 2]
    if phase2:
        checkpoints.append((variant["name"], Path(phase2[-1]["global"])))

cmd = [
    str(PYTHON_BIN),
    str(EVAL_SCRIPT),
    "--workspace",
    str(EVAL_WORKSPACE),
    "--splits",
    EVAL_SPLITS,
    "--batch-size",
    str(EVAL_BATCH_SIZE),
    "--no-plots",
    "--verbose",
]
if EVAL_DEVICE:
    cmd.extend(["--device", EVAL_DEVICE])
for label, path in checkpoints:
    cmd.extend(["--checkpoint", f"{label}={path}"])

print("eval_workspace:", EVAL_WORKSPACE)
print("checkpoints:")
for label, path in checkpoints:
    print(" ", label, path)
print(" ".join(cmd))

if RUN_EVAL:
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)
else:
    print("RUN_EVAL=False; set True after training finishes.")


## 9. Read Evaluation Tables

In [ ]:
report_root = EVAL_WORKSPACE / "validation_reports"
summary_csv = report_root / "paper_protocol_eval_summary.csv"
classwise_csv = report_root / "paper_protocol_classwise_summary.csv"

if summary_csv.exists():
    summary = pd.read_csv(summary_csv)
    cols = ["checkpoint_label", "split", "precision", "recall", "map50", "map50_95"]
    display(summary[cols].sort_values(["split", "map50"], ascending=[True, False]))
    total = summary[summary["split"].eq("scene_total")].copy()
    if not total.empty:
        display(total[cols].sort_values("map50", ascending=False))
else:
    print("No summary yet:", summary_csv)

if classwise_csv.exists():
    classwise = pd.read_csv(classwise_csv)
    display(classwise[classwise["split"].eq("scene_total")].sort_values(["class", "map50_95"], ascending=[True, False]).head(120))
else:
    print("No classwise summary yet:", classwise_csv)


## 10. DQA Update-Gating Logs

In [ ]:
rows = []
for variant in selected:
    state_path = variant_work_root(variant) / "dqa_cwa_state.json"
    if not state_path.exists():
        continue
    state = json.loads(state_path.read_text(encoding="utf-8"))
    for row in state.get("update_gating", []):
        rows.append({"variant": variant["name"], **row})

gate_df = pd.DataFrame(rows)
if not gate_df.empty:
    display(gate_df.tail(80))
    display(gate_df.groupby("variant")[["residual_blend", "min_server_alpha", "classwise_blend", "server_anchor"]].agg(["min", "max"]))
else:
    print("No update_gating entries yet.")

for variant in selected:
    threshold_log = variant_stats_root(variant) / "update_gating_policy_schedule.jsonl"
    print("\n", variant["name"], threshold_log, "exists=", threshold_log.exists())
    if threshold_log.exists():
        records = [json.loads(line) for line in threshold_log.read_text(encoding="utf-8").splitlines() if line.strip()]
        df = pd.DataFrame(records)
        if not df.empty:
            compact = df[["phase", "round", "client_id", "enabled", "reason", "nms_conf_thres", "teacher_loss_weight"]].copy()
            compact["low_min"] = df["ignore_thres_low"].map(lambda xs: min(xs) if isinstance(xs, list) else None)
            compact["high_min"] = df["ignore_thres_high"].map(lambda xs: min(xs) if isinstance(xs, list) else None)
            display(compact.tail(10))
